In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/인공지능사관학교 6기/Agust_Project

/content/drive/MyDrive/인공지능사관학교 6기/Agust_Project


In [ ]:
# Libraries
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
pd.set_option('display.max_rows', None)

In [82]:
data = pd.read_csv('./data/제주도방언데이터.csv')

In [ ]:
data.columns

Index(['standard_form', 'dialect_form'], dtype='object')

In [ ]:
def clean_duplicate_data(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:

    # 1. 완전 중복 행 제거 (standard_form과 dialect_form 모두 동일한 경우)
    initial_clean_df = df.drop_duplicates(subset=['standard_form', 'dialect_form'], keep='first')
    print("완전 중복 제거 후 데이터셋 크기:", len(initial_clean_df))

    # 2. dialect_form만 중복되는 행 식별 (수동 검토 대상)
    # dialect_form은 같지만 standard_form이 다른 케이스를 찾아냅니다.
    dialect_duplicates_mask = initial_clean_df.duplicated(subset=['dialect_form'], keep=False)
    dialect_review_df = initial_clean_df[dialect_duplicates_mask].sort_values(by='dialect_form')
    print("수동 검토가 필요한 잠재적 중복 방언 데이터 수:", len(dialect_review_df))

    # 3. standard_form만 중복되는 행 식별 (수동 검토 대상)
    # standard_form은 같지만 dialect_form이 다른 케이스를 찾아냅니다.
    standard_duplicates_mask = initial_clean_df.duplicated(subset=['standard_form'], keep=False)
    standard_review_df = initial_clean_df[standard_duplicates_mask].sort_values(by='standard_form')
    print("수동 검토가 필요한 잠재적 중복 표준어 데이터 수:", len(standard_review_df))

    # 4.최종 검토 대상 데이터셋
    combined_review_mask = standard_duplicates_mask | dialect_duplicates_mask
    review_df = pd.concat([dialect_review_df, standard_review_df], axis=0)
    print("수동 검토가 필요한 잠재적 중복 데이터 수:", len(review_df))

    # 5. 중복, 검토 대상을 제외한 순수 데이터셋
    clean_df = initial_clean_df[~combined_review_mask]
    print("4단계: 최종적으로 Clean한 데이터셋 크기:", len(clean_df))

    return dialect_review_df, standard_review_df, clean_df

In [83]:
dialect_review_df, standard_review_df, clean_df = clean_duplicate_data(data)

완전 중복 제거 후 데이터셋 크기: 234019
수동 검토가 필요한 잠재적 중복 방언 데이터 수: 82
수동 검토가 필요한 잠재적 중복 표준어 데이터 수: 391
수동 검토가 필요한 잠재적 중복 데이터 수: 473
4단계: 최종적으로 Clean한 데이터셋 크기: 233562


In [84]:
dialect_review_df = dialect_review_df.reset_index(drop=True)

In [85]:
standard_review_df = standard_review_df.reset_index(drop=True)

In [86]:
clean_df = clean_df.reset_index(drop=True)

In [ ]:
for name, group in dialect_review_df.groupby('dialect_form'):
    print(f"--- 방언: '{name}' ---")
    print(group)
    print("\n")

--- 방언: 'ᄂᆞ물 ᄉᆞᆱ앙도 먹고 .' ---
  standard_form    dialect_form
0   나물 삶아도 먹고 .  ᄂᆞ물 ᄉᆞᆱ앙도 먹고 .
1  나물 삶아서도 먹고 .  ᄂᆞ물 ᄉᆞᆱ앙도 먹고 .


--- 방언: 'ᄎᆞᆯ레에 대해서 말씀해 주십시오 .' ---
          standard_form          dialect_form
2    반찬에 대해서 말씀해 주십시오 .  ᄎᆞᆯ레에 대해서 말씀해 주십시오 .
3  ᄎᆞᆯ레에 대해서 말씀해 주십시오 .  ᄎᆞᆯ레에 대해서 말씀해 주십시오 .


--- 방언: 'ᄒᆞᆫ 번 더 ᄀᆞᆯ아줍서 .' ---
    standard_form      dialect_form
4   한 번 더 갈아주세요 .  ᄒᆞᆫ 번 더 ᄀᆞᆯ아줍서 .
5  한 번 더 말해보십시오 .  ᄒᆞᆫ 번 더 ᄀᆞᆯ아줍서 .


--- 방언: '갈 때 무신거 헹 가 ?' ---
     standard_form   dialect_form
6     갈 때 뭐 해서 가 ?  갈 때 무신거 헹 가 ?
7  갈 때 무엇을 해서 가요 ?  갈 때 무신거 헹 가 ?


--- 방언: '건 무신거엔 ᄀᆞᆯ아 ?' ---
  standard_form   dialect_form
8   그건 뭐라고 말해 ?  건 무신거엔 ᄀᆞᆯ아 ?
9  건 무엇이라고 말해 ?  건 무신거엔 ᄀᆞᆯ아 ?


--- 방언: '것ᄀᆞ라 뭐엔 ᄀᆞᆯ아 ?' ---
     standard_form    dialect_form
10    것보고 뭐라고 말해 ?  것ᄀᆞ라 뭐엔 ᄀᆞᆯ아 ?
11  그것 보고 뭐라고 말해 ?  것ᄀᆞ라 뭐엔 ᄀᆞᆯ아 ?


--- 방언: '겅헹 어떵헤마씨 ?' ---
      standard_form dialect_form
12   그렇게 해서 어떻게해요 ?   겅헹 어떵헤마씨 ?
13  그렇게 해서 어떻게 해요 ?   겅헹 어떵헤마씨 ?


--- 방언: '그거 무신거엔 ᄀᆞᆯ아 ?' ---
    standar

In [ ]:
# !pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.1 MB/s eta 0:00:00


In [87]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipywidgets import HBox, VBox
from ipywidgets import GridspecLayout

def create_review_ui(df: pd.DataFrame) -> tuple[GridspecLayout, widgets.Button, widgets.Output, list]:
    """
    데이터프레임을 입력받아 수동 검토를 위한 인터랙티브 UI를 생성합니다.

    Args:
        df: 'standard_form'과 'dialect_form' 컬럼을 포함하는 pandas DataFrame.

    Returns:
        GridspecLayout, Button, Output, Checkbox 리스트를 담은 튜플.
    """

    # 1. UI 요소 생성
    checkboxes = [widgets.Checkbox(value=False) for _ in range(len(df))]
    save_button = widgets.Button(description="선택된 행 저장")
    output = widgets.Output()

    # 2. GridspecLayout 객체 생성 및 헤더 배치
    grid_layout = GridspecLayout(len(df) + 1, 3, width='100%')
    grid_layout[0, 0] = widgets.Label(value="선택")
    grid_layout[0, 1] = widgets.Label(value="표준어")
    grid_layout[0, 2] = widgets.Label(value="방언")

    # 3. 데이터 행에 위젯 배치
    for i, row in df.iterrows():
        row_index = i + 1
        grid_layout[row_index, 0] = checkboxes[i]
        grid_layout[row_index, 1] = widgets.Label(value=row['standard_form'])
        grid_layout[row_index, 2] = widgets.Label(value=row['dialect_form'])

    # 4. 버튼 클릭 이벤트 처리
    def on_button_clicked(b):
        with output:
            clear_output()
            selected_indices = [i for i, checkbox in enumerate(checkboxes) if checkbox.value]
            selected_df = df.iloc[selected_indices]

            print("--- 선택된 데이터 ---")
            display(selected_df)
            print("\n선택된 행의 인덱스:", selected_indices)

    save_button.on_click(on_button_clicked)

    return grid_layout, save_button, output, checkboxes

In [88]:
# Dialect Review UI 생성 및 표시
print("--- 제주 방언 중복 검토 ---")
dialect_grid, dialect_button, dialect_output, dialect_checkboxes = create_review_ui(dialect_review_df)
display(dialect_grid, dialect_button, dialect_output)

--- 제주 방언 중복 검토 ---


GridspecLayout(children=(Label(value='선택', layout=Layout(grid_area='widget001')), Label(value='표준어', layout=La…

Button(description='선택된 행 저장', style=ButtonStyle())

Output()

In [89]:
dialect_indicies = [1, 2, 5, 6, 8, 11, 13, 15, 16, 19, 20, 23, 25, 26, 29, 30, 33, 36, 37, 40, 43, 47, 49, 53, 54, 58, 61, 62, 65, 67, 74, 80]

In [90]:
reviewed_dialect_df = dialect_review_df.iloc[dialect_indicies]

In [ ]:
# Standard Review UI 생성 및 표시
print("\n--- 표준어 중복 검토 ---")
standard_grid, standard_button, standard_output, standard_checkboxes = create_review_ui(standard_review_df)
display(standard_grid, standard_button, standard_output)


--- 표준어 중복 검토 ---


GridspecLayout(children=(Label(value='선택', layout=Layout(grid_area='widget001')), Label(value='표준어', layout=La…

Button(description='선택된 행 저장', style=ButtonStyle())

Output()

In [ ]:
# standard_review_df.to_csv('./data/standard_review_data.csv', index=False, encoding='utf-8-sig')

In [91]:
reviewed_standard_df = pd.read_csv('./data/reviewed_standard_data.csv')

In [92]:
final_df = pd.concat([clean_df, reviewed_dialect_df, reviewed_standard_df], axis=0).reset_index(drop=True)

In [93]:
a, b, c = clean_duplicate_data(final_df)

완전 중복 제거 후 데이터셋 크기: 233766
수동 검토가 필요한 잠재적 중복 방언 데이터 수: 2
수동 검토가 필요한 잠재적 중복 표준어 데이터 수: 4
수동 검토가 필요한 잠재적 중복 데이터 수: 6
4단계: 최종적으로 Clean한 데이터셋 크기: 233760


In [94]:
a

,standard_form,dialect_form
233684,"""아 , 그랬구나 .""","""아 , 겅헷구나예 ."""
233685,"""아 , 그랬군요 .""","""아 , 겅헷구나예 ."""


In [81]:
b

,standard_form,dialect_form
233570,그건 무슨 말입니까 ?,그건 무신 말이꽈 ?
233610,그건 무슨 말입니까 ?,그건 뭔 말이꽈 ?
233566,그건 뭐라고 말해 ?,건 무신거엔 ᄀᆞᆯ아 ?
233611,그건 뭐라고 말해 ?,건 뭐엔 ᄀᆞᆯ아 ?


In [96]:
final_df = final_df.drop(index=[233684, 233610, 233566]).reset_index(drop=True)

In [97]:
d, e, f = clean_duplicate_data(final_df)

완전 중복 제거 후 데이터셋 크기: 233763
수동 검토가 필요한 잠재적 중복 방언 데이터 수: 0
수동 검토가 필요한 잠재적 중복 표준어 데이터 수: 0
수동 검토가 필요한 잠재적 중복 데이터 수: 0
4단계: 최종적으로 Clean한 데이터셋 크기: 233763


In [99]:
# final_df.to_csv('./data/duplicate_dropped_jeju_data.csv', index=False, encoding='utf-8-sig')

In [101]:
final_df.iloc[1406]

,1406
standard_form,"아 , 그건 뭐라고 하더라 . 나도 그 당시에도 뿔 모지랑이 , 뿔 모지랑이라고 하지"
dialect_form,"아 , 그건 뭐렌 허드라 . 나도 그 당시에도 뿔 몽그레기 , 뿔 몽그레기옌 허주게"


In [102]:
data = pd.read_csv('./data/duplicate_dropped_jeju_data.csv')

In [107]:
data.iloc[1401]

,1401
standard_form,응 . 그 거시기 하니까 그 사람들 그만 다 망가져 오죽해야 .
dialect_form,ᄋᆞ게 . 그 거시기 허난게 그 사람덜 오 다 망가져 오죽해샤 .
